# OR · 07 Safety Stock Intro



## 📋 Contexto del Caso de Negocio

**Empresa:** "RetailCorp Distribución" - Cadena de retail con 50+ tiendas y centro de distribución centralizado.

**Situación actual:**
- **Stock de seguridad actual**: Calculado como "2-3 semanas de stock" sin rigor estadístico
- **Problema:** Nivel de servicio resultante es incierto (puede oscilar entre 85% y 99% sin control)
- Factores relevantes:
  - Variabilidad alta en demanda (CV promedio: 0.45)
  - Lead time de proveedores: 4-10 días según categoría
  - Costos de almacenamiento: 18% anual sobre valor de inventario

**Impacto financiero:**
- Stockouts frecuentes en productos críticos (pérdida estimada: $250K/año)
- Exceso de inventario en productos de baja rotación (capital inmovilizado: $1.2M)
- Nivel de servicio promedio: 91% (objetivo: 95% categoría A, 92% categoría B)

**Objetivo:** Implementar metodología estadística de cálculo de stock de seguridad para:
1. Garantizar niveles de servicio objetivo por categoría de producto
2. Optimizar capital de trabajo reduciendo inventario excedente
3. Establecer políticas de reorden basadas en datos (ROP = Reorder Point)
4. Capacitar al equipo de planificación en conceptos estadísticos

### 💼 ¿Por qué es IMPORTANTE?
- **Balancear costo vs servicio:** Trade-off entre tener stock (costo) y evitar rupturas (servicio al cliente)
- **Fundamento matemático:** Reemplazar reglas empíricas por cálculo basado en distribuciones estadísticas
- **Diferenciación por producto:** No todos los SKUs requieren el mismo nivel de protección
- **Visibilidad de inventario:** Entender cuántos días de demanda cubrimos con el safety stock

### 🎁 ¿PARA QUÉ sirve?
- **Entrenamiento interno:** Educar equipos de supply chain en conceptos estadísticos
- **Comunicación con procurement:** Justificar decisiones de inventario con datos
- **Validación de políticas:** Auditar políticas existentes y detectar gaps
- **Base para optimización:** Sentar fundamentos para modelos avanzados (multi-echelon, variabilidad de LT)

### 🔧 ¿CÓMO se implementa?
- **Datos requeridos:** Historial de órdenes (demand), productos (categoría), inventario actual
- **Cálculo principal:** `Safety Stock = Z × σ_demanda × √(Lead Time)` donde Z es z-score del nivel de servicio
- **Métrica resultado:** `ROP = Demanda_LT + Safety_Stock` (punto de reorden)
- **Técnica aplicada:** Distribución normal estándar, análisis de variabilidad (CV = std/mean)

---

## 🎯 Objetivos de Aprendizaje

- Comprender los conceptos fundamentales de incertidumbre en demanda y variabilidad
- Calcular stock de seguridad para diferentes niveles de servicio objetivo
- Visualizar el impacto de la variabilidad en las políticas de inventario
- Establecer puntos de reorden (ROP) basados en análisis estadístico

## 📦 Instalación de Librerías Necesarias

**Antes de ejecutar este notebook, asegúrate de tener instaladas todas las dependencias.**

### Opción 1: Instalación dentro del notebook
Ejecuta la siguiente celda para instalar las librerías necesarias:

```python
%pip install pandas numpy plotly scipy
```

### Opción 2: Instalación desde terminal
Si prefieres instalar desde la terminal, ejecuta:

```bash
# PowerShell o CMD
pip install pandas numpy plotly scipy

# O si usas el proyecto completo con pyproject.toml
pip install -e .[core,notebooks,or]
```

### Librerías requeridas:
- `pandas`: Manipulación y análisis de datos
- `numpy`: Cálculos numéricos y arrays
- `plotly`: Visualización interactiva
- `scipy`: Funciones estadísticas (distribución normal, z-scores)

---

### 📝 Información del Notebook

| Campo | Valor |
| :--- | :--- |
| **🆔 ID** | `OR-07` |
| **📛 Título** | `Safety Stock Introduction` |
| **🔹 Especialidad** | `Operations Research / Inventory Optimization` |
| **⚙️ Proceso** | `Plan` |
| **🧠 Nivel** | `Basic` |
| **⏱️ Duración** | `30 min` |
| **🏷️ Etiquetas** | `safety-stock`, `inventory`, `service-level`, `statistical-methods` |

---

## ⚙️ Configuración Inicial

## 🎯 Contexto del Notebook

### ¿Qué?
Introducción a los conceptos fundamentales de stock de seguridad: z-score, demanda durante lead time, variabilidad (CV), y cálculo de punto de reorden (ROP). Se implementa la fórmula clásica con ejemplos ilustrativos.

### ¿Por qué?
Las empresas suelen calcular safety stock empíricamente ("2 semanas de stock") sin rigor estadístico, resultando en niveles de servicio inciertos. Este notebook educa en metodología probabilística para tomar decisiones basadas en datos.

### ¿Para qué?
- Entrenamiento interno de equipos de supply chain
- Comunicación efectiva con procurement y stakeholders
- Validación y auditoría de políticas de inventario existentes
- Base conceptual para notebooks avanzados (OR-01, OR-02)

### ¿Cuándo?
- Contenido educativo para nuevos integrantes del equipo
- Pre-auditoría de políticas de inventario
- Revisión trimestral de niveles de servicio
- Antes de implementar modelos avanzados de optimización

### ¿Cómo?
1. Cargar datos históricos de demanda y productos
2. Calcular estadísticas de demanda (media, desviación estándar, CV)
3. Definir parámetros de política (nivel de servicio, lead time)
4. Aplicar fórmula de safety stock: `SS = Z × σ × √LT`
5. Calcular punto de reorden: `ROP = Demanda_LT + SS`
6. Comparar con inventario actual e identificar gaps

In [17]:
# ⚙️ Configuración de rutas
import sys
from pathlib import Path

def resolve_repo_root():
    """Detecta raíz del repositorio buscando carpetas data/ y notebooks/"""
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / 'data').exists() and (parent / 'notebooks').exists():
            return parent
    return current

root = resolve_repo_root()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"✅ Rutas configuradas: {root}")

✅ Rutas configuradas: f:\GitHub\supply-chain-data-notebooks


In [18]:
# 📚 Importar librerías
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuración de rutas
DATA_DIR = root / "data" / "raw"
OUTPUT_DIR = root / "data" / "processed" / "or07"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Librerías cargadas")
print(f"📁 Directorio datos: {DATA_DIR.resolve()}")
print(f"📁 Directorio salida: {OUTPUT_DIR.resolve()}")

✅ Librerías cargadas
📁 Directorio datos: F:\GitHub\supply-chain-data-notebooks\data\raw
📁 Directorio salida: F:\GitHub\supply-chain-data-notebooks\data\processed\or07


---

# 🔧 PASOS DEL NOTEBOOK

---

## 📊 Paso 1: Cargar y Preparar Datos

**Técnica:** Ingesta de datos desde fuentes raw (CSV)

**Parámetros clave:**
- `parse_dates`: Columnas a convertir a datetime (date)
- Datasets: orders.csv, products.csv, inventory.csv

In [19]:
# Cargar datos
try:
    df_orders = pd.read_csv(DATA_DIR / "orders.csv", parse_dates=['date'])
    df_products = pd.read_csv(DATA_DIR / "products.csv")
    df_inventory = pd.read_csv(DATA_DIR / "inventory.csv")
    
    print("📊 Datos cargados desde archivos CSV:")
    print(f"  - Órdenes: {len(df_orders):,} registros")
    print(f"  - Productos: {len(df_products):,} SKUs")
    print(f"  - Inventario: {len(df_inventory):,} registros")
    
except FileNotFoundError:
    print("⚠️ Archivos no encontrados. Generando datos sintéticos de ejemplo...\n")
    
    # Generar datos sintéticos realistas para demostración
    np.random.seed(42)
    
    # 1. Productos (50 SKUs representativos)
    categories = ['Electronics', 'PersonalCare', 'Household', 'Snacks']
    skus = [f'SKU-{i:04d}' for i in range(1, 51)]
    df_products = pd.DataFrame({
        'sku': skus,
        'category': np.random.choice(categories, 50),
        'brand': [f'Brand-{chr(65+i%10)}' for i in range(50)],
        'unit_cost': np.random.uniform(5, 200, 50).round(2)
    })
    
    # 2. Órdenes (90 días de historial con patrones realistas)
    date_range = pd.date_range(end=pd.Timestamp.today(), periods=90, freq='D')
    orders_data = []
    
    for sku in skus:
        # Demanda base según categoría
        category = df_products[df_products['sku'] == sku]['category'].values[0]
        if category == 'Electronics':
            base_demand = np.random.uniform(15, 35)
            variability = 0.4  # Alta variabilidad
        elif category == 'PersonalCare':
            base_demand = np.random.uniform(40, 80)
            variability = 0.3  # Moderada
        elif category == 'Household':
            base_demand = np.random.uniform(25, 60)
            variability = 0.35
        else:  # Snacks
            base_demand = np.random.uniform(50, 100)
            variability = 0.25  # Baja variabilidad
        
        for date in date_range:
            # Simular demanda diaria con estacionalidad semanal
            day_of_week = date.dayofweek
            weekend_factor = 1.3 if day_of_week in [5, 6] else 1.0
            
            daily_qty = int(np.random.normal(
                base_demand * weekend_factor,
                base_demand * variability
            ))
            daily_qty = max(0, daily_qty)  # No negativo
            
            if daily_qty > 0:
                orders_data.append({
                    'order_id': f'ORD-{len(orders_data)+1:06d}',
                    'date': date,
                    'sku': sku,
                    'qty': daily_qty
                })
    
    df_orders = pd.DataFrame(orders_data)
    
    # 3. Inventario actual (niveles realistas)
    current_inventory = []
    for sku in skus:
        avg_demand = df_orders[df_orders['sku'] == sku]['qty'].mean()
        # Inventario entre 5-15 días de demanda
        current_stock = int(np.random.uniform(5, 15) * avg_demand)
        current_inventory.append({
            'sku': sku,
            'on_hand': current_stock,
            'warehouse': 'WH-01'
        })
    
    df_inventory = pd.DataFrame(current_inventory)
    
    print("✅ Datos sintéticos generados:")
    print(f"  - Órdenes: {len(df_orders):,} registros (90 días)")
    print(f"  - Productos: {len(df_products):,} SKUs")
    print(f"  - Inventario: {len(df_inventory):,} registros")
    print(f"\n💡 Características realistas:")
    print(f"  • Variabilidad por categoría (Electronics > Household > PersonalCare > Snacks)")
    print(f"  • Estacionalidad semanal (picos en fin de semana)")
    print(f"  • Lead times diferenciados por categoría")

print(f"\n📋 Muestra de órdenes:")
display(df_orders.head(3))

print(f"\n📦 Muestra de productos:")
display(df_products.head(3))

📊 Datos cargados desde archivos CSV:
  - Órdenes: 8,504 registros
  - Productos: 200 SKUs
  - Inventario: 3,000 registros

📋 Muestra de órdenes:


,order_id,date,sku,qty,location_id,channel
0,ORD-100000,2024-01-01,SKU-00023,13,LOC-013,Retail
1,ORD-100001,2024-01-01,SKU-00111,7,LOC-011,B2B
2,ORD-100002,2024-01-01,SKU-00100,5,LOC-019,Ecom



📦 Muestra de productos:


,sku,category,brand,unit_cost
0,SKU-00001,Household,BrandB,56.62
1,SKU-00002,Electronics,BrandC,114.89
2,SKU-00003,PersonalCare,BrandA,7.09


---

## 🔄 Paso 2: Agregación de Demanda Diaria

**Técnica:** Agregación temporal de datos transaccionales

**Objetivo:** Consolidar órdenes por SKU y fecha para análisis de variabilidad diaria

In [20]:
# Agregar demanda diaria por SKU
df_daily_demand = df_orders.groupby(['sku', 'date'])['qty'].sum().reset_index()
df_daily_demand.rename(columns={'qty': 'daily_demand'}, inplace=True)

print("📅 Demanda diaria agregada")
print(f"Total registros: {len(df_daily_demand):,}")
print(f"Rango de fechas: {df_daily_demand['date'].min().date()} a {df_daily_demand['date'].max().date()}")
print(f"Días con datos: {df_daily_demand['date'].nunique()}")

display(df_daily_demand.head())

# Distribución de demanda para un SKU ejemplo con alta demanda
sample_sku = df_daily_demand.groupby('sku')['daily_demand'].sum().idxmax()
sample_data = df_daily_demand[df_daily_demand['sku'] == sample_sku]
sample_mean = sample_data['daily_demand'].mean()
sample_std = sample_data['daily_demand'].std()

print(f"\n📊 Análisis de SKU ejemplo: {sample_sku}")
print(f"  • Demanda promedio: {sample_mean:.1f} unidades/día")
print(f"  • Desviación estándar: {sample_std:.1f} unidades")
print(f"  • CV: {(sample_std/sample_mean):.2f} ({'Alta' if sample_std/sample_mean > 0.5 else 'Moderada' if sample_std/sample_mean > 0.3 else 'Baja'} variabilidad)")

fig = px.histogram(
    sample_data, x='daily_demand', 
    title=f"Distribución de Demanda Diaria - {sample_sku}<br><sub>μ={sample_mean:.1f}, σ={sample_std:.1f}, CV={sample_std/sample_mean:.2f}</sub>",
    labels={'daily_demand': 'Demanda Diaria (unidades)', 'count': 'Frecuencia (días)'},
    nbins=20,
    color_discrete_sequence=['steelblue']
)

# Agregar línea de media
fig.add_vline(x=sample_mean, line_dash="dash", line_color="red", 
              annotation_text=f"Media: {sample_mean:.1f}",
              annotation_position="top")

fig.update_layout(height=400, showlegend=False)
fig.show()

📅 Demanda diaria agregada
Total registros: 6,700
Rango de fechas: 2024-01-01 a 2024-03-31
Días con datos: 91


,sku,date,daily_demand
0,SKU-00001,2024-01-10,9
1,SKU-00001,2024-01-13,8
2,SKU-00001,2024-01-15,5
3,SKU-00001,2024-01-16,7
4,SKU-00001,2024-01-18,7



📊 Análisis de SKU ejemplo: SKU-00081
  • Demanda promedio: 16.0 unidades/día
  • Desviación estándar: 13.5 unidades
  • CV: 0.85 (Alta variabilidad)


---

## 📏 Paso 3: Cálculo de Variabilidad de Demanda

**Concepto:** Coeficiente de Variación (CV) como medida de imprevisibilidad

**Fórmula:** `CV = σ / μ` (desviación estándar / media)

**Interpretación de CV:**
- **CV < 0.3**: Demanda estable, predecible
- **0.3 ≤ CV ≤ 0.7**: Demanda moderadamente variable
- **CV > 0.7**: Alta variabilidad → considerar mayor safety stock

**Caso de uso:** Cuantificar incertidumbre por SKU para ajustar políticas de inventario

In [21]:
# Calcular estadísticas de demanda por SKU
demand_stats = df_daily_demand.groupby('sku')['daily_demand'].agg([
    ('avg_demand', 'mean'),
    ('std_demand', 'std'),
    ('min_demand', 'min'),
    ('max_demand', 'max'),
    ('cv', lambda x: x.std() / x.mean() if x.mean() > 0 else 0)  # Coeficiente de variación
]).reset_index()

# Enriquecer con información de producto
demand_stats = demand_stats.merge(df_products[['sku', 'category', 'brand']], on='sku', how='left')

# Clasificar por nivel de variabilidad
demand_stats['variability_class'] = pd.cut(
    demand_stats['cv'],
    bins=[0, 0.3, 0.7, float('inf')],
    labels=['Baja (<0.3)', 'Moderada (0.3-0.7)', 'Alta (>0.7)']
)

print("📊 Estadísticas de Demanda por SKU:")
display(demand_stats.head(10))

print(f"\n📈 Análisis de Variabilidad:")
print(f"  • CV promedio: {demand_stats['cv'].mean():.3f}")
print(f"  • CV mediana: {demand_stats['cv'].median():.3f}")
print(f"  • Rango CV: [{demand_stats['cv'].min():.3f}, {demand_stats['cv'].max():.3f}]")

print(f"\n🎯 Distribución por Clase de Variabilidad:")
variability_counts = demand_stats['variability_class'].value_counts()
for var_class, count in variability_counts.items():
    pct = (count / len(demand_stats)) * 100
    print(f"  • {var_class}: {count} SKUs ({pct:.1f}%)")

print(f"\n💡 Implicación de Negocio:")
high_var_count = len(demand_stats[demand_stats['cv'] > 0.7])
if high_var_count > 0:
    print(f"  ⚠️ {high_var_count} SKUs tienen alta variabilidad (CV > 0.7)")
    print(f"     → Requerirán mayor safety stock para mantener nivel de servicio")
    print(f"     → Considerar pronósticos mejorados o políticas dinámicas")

📊 Estadísticas de Demanda por SKU:


,sku,avg_demand,std_demand,min_demand,max_demand,cv,category,brand,variability_class
0,SKU-00001,12.933333,9.373232,0,44,0.724734,Household,BrandB,Alta (>0.7)
1,SKU-00002,11.888889,11.000721,0,62,0.925294,Electronics,BrandC,Alta (>0.7)
2,SKU-00003,10.909091,8.063667,1,33,0.739169,PersonalCare,BrandA,Alta (>0.7)
3,SKU-00004,14.604651,10.659535,0,44,0.729873,Electronics,BrandA,Alta (>0.7)
4,SKU-00005,14.024390,9.903251,3,40,0.706145,Electronics,BrandD,Alta (>0.7)
5,SKU-00006,11.928571,7.736056,1,29,0.648532,Snacks,BrandC,Moderada (0.3-0.7)
6,SKU-00007,12.176471,9.771544,1,41,0.802494,PersonalCare,BrandE,Alta (>0.7)
7,SKU-00008,10.642857,8.438072,0,33,0.792839,PersonalCare,BrandC,Alta (>0.7)
8,SKU-00009,8.733333,5.105327,3,24,0.584579,PersonalCare,BrandD,Moderada (0.3-0.7)
9,SKU-00010,10.031250,9.740121,1,47,0.970978,Electronics,BrandD,Alta (>0.7)



📈 Análisis de Variabilidad:
  • CV promedio: 0.777
  • CV mediana: 0.767
  • Rango CV: [0.538, 1.042]

🎯 Distribución por Clase de Variabilidad:
  • Alta (>0.7): 151 SKUs (75.5%)
  • Moderada (0.3-0.7): 49 SKUs (24.5%)
  • Baja (<0.3): 0 SKUs (0.0%)

💡 Implicación de Negocio:
  ⚠️ 151 SKUs tienen alta variabilidad (CV > 0.7)
     → Requerirán mayor safety stock para mantener nivel de servicio
     → Considerar pronósticos mejorados o políticas dinámicas


---

## ⚙️ Paso 4: Parámetros de Política de Inventario

**Concepto:** Definición de parámetros clave para cálculo de safety stock

**Parámetros de negocio:**
- `SERVICE_LEVEL`: Nivel de servicio objetivo (0-1)
- `LEAD_TIME_DAYS`: Días de lead time de reabastecimiento
- `Z-score`: Cuantil de distribución normal para el service level

In [22]:
# Parámetros de negocio (baseline)
SERVICE_LEVEL = 0.95  # 95% nivel de servicio
LEAD_TIME_DAYS = 7    # 7 días lead time de reabastecimiento

# Z-score para nivel de servicio
z_score = stats.norm.ppf(SERVICE_LEVEL)

print(f"🎯 Parámetros de Política (Baseline):")
print(f"  • Nivel de servicio objetivo: {SERVICE_LEVEL*100:.1f}%")
print(f"  • Z-score correspondiente: {z_score:.4f}")
print(f"  • Lead time de reabastecimiento: {LEAD_TIME_DAYS} días")

print(f"\n📚 Interpretación de Nivel de Servicio:")
print(f"  ✅ {SERVICE_LEVEL*100:.1f}% de los ciclos de reabastecimiento NO tendrán stockout")
print(f"  ⚠️ {(1-SERVICE_LEVEL)*100:.1f}% de los ciclos PODRÍAN tener stockout")

print(f"\n💰 Tabla de Referencia Z-scores:")
reference_levels = [0.90, 0.95, 0.975, 0.99, 0.995]
print(f"  {'Nivel de Servicio':<20} {'Z-score':<10} {'Riesgo Stockout'}")
print(f"  {'-'*50}")
for sl in reference_levels:
    z = stats.norm.ppf(sl)
    risk = 1 - sl
    marker = '👉' if abs(sl - SERVICE_LEVEL) < 0.001 else '  '
    print(f"  {marker} {sl*100:>5.1f}% {' '*10} {z:>7.4f} {' '*5} {risk*100:>5.1f}%")

print(f"\n🎓 Regla práctica:")
print(f"  • Cada aumento de ~0.3 en Z-score reduce riesgo de stockout a la mitad")
print(f"  • Pero también aumenta el safety stock proporcionalmente")

🎯 Parámetros de Política (Baseline):
  • Nivel de servicio objetivo: 95.0%
  • Z-score correspondiente: 1.6449
  • Lead time de reabastecimiento: 7 días

📚 Interpretación de Nivel de Servicio:
  ✅ 95.0% de los ciclos de reabastecimiento NO tendrán stockout
  ⚠️ 5.0% de los ciclos PODRÍAN tener stockout

💰 Tabla de Referencia Z-scores:
  Nivel de Servicio    Z-score    Riesgo Stockout
  --------------------------------------------------
      90.0%             1.2816        10.0%
  👉  95.0%             1.6449         5.0%
      97.5%             1.9600         2.5%
      99.0%             2.3263         1.0%
      99.5%             2.5758         0.5%

🎓 Regla práctica:
  • Cada aumento de ~0.3 en Z-score reduce riesgo de stockout a la mitad
  • Pero también aumenta el safety stock proporcionalmente


---

## 🛡️ Paso 5: Fórmula de Stock de Seguridad

**Concepto:** Cálculo del buffer de inventario para proteger contra variabilidad de demanda

**Fórmula Clásica:**

$$
\text{Safety Stock} = Z \times \sigma_{\text{demanda}} \times \sqrt{\text{Lead Time}}
$$

**Donde:**
- **Z**: Z-score del nivel de servicio (ej: 1.65 para 95%, 1.96 para 97.5%, 2.33 para 99%)
- **σ_demanda**: Desviación estándar de demanda diaria por SKU
- **Lead Time**: Días hasta reabastecer

**Interpretación de negocio:**
- **Z ↑** → Más servicio → Más stock de seguridad → Menos rupturas (pero más capital invertido)
- **σ ↑** → Demanda más volátil → Más stock de seguridad necesario
- **LT ↑** → Más días cubiertos → Más stock de seguridad requerido

**Buenas prácticas:**
- Usar ventanas móviles (últimos 60-90 días) para calcular σ
- Ajustar Z por criticidad de producto (Clase A: 98-99%, B: 95%, C: 90-92%)
- Revisar nivel de servicio trimestral según fill rate real
- Considerar estacionalidad al calcular estadísticas

**Aplicación:** Productos con demanda aleatoria y lead time relativamente constante

In [23]:
# Calcular stock de seguridad
demand_stats['safety_stock'] = (
    z_score * demand_stats['std_demand'] * np.sqrt(LEAD_TIME_DAYS)
).round(0).astype(int)

# Calcular punto de reorden (ROP = demanda durante lead time + safety stock)
demand_stats['reorder_point'] = (
    (demand_stats['avg_demand'] * LEAD_TIME_DAYS) + demand_stats['safety_stock']
).round(0).astype(int)

# Días de cobertura del safety stock
demand_stats['coverage_days'] = (
    demand_stats['safety_stock'] / demand_stats['avg_demand']
).round(1)

print("🛡️  Stock de Seguridad Calculado:")
display(demand_stats[[
    'sku', 'category', 'avg_demand', 'std_demand', 
    'safety_stock', 'reorder_point', 'coverage_days'
]])

🛡️  Stock de Seguridad Calculado:


,sku,category,avg_demand,std_demand,safety_stock,reorder_point,coverage_days
0,SKU-00001,Household,12.933333,9.373232,41,132,3.2
1,SKU-00002,Electronics,11.888889,11.000721,48,131,4.0
2,SKU-00003,PersonalCare,10.909091,8.063667,35,111,3.2
3,SKU-00004,Electronics,14.604651,10.659535,46,148,3.1
4,SKU-00005,Electronics,14.024390,9.903251,43,141,3.1
...,...,...,...,...,...,...,...
195,SKU-00196,PersonalCare,12.812500,11.882319,52,142,4.1
196,SKU-00197,PersonalCare,9.676471,8.303780,36,104,3.7
197,SKU-00198,Beverages,10.160000,6.780364,30,101,3.0
198,SKU-00199,Electronics,10.600000,7.452595,32,106,3.0


---

## 📅 Paso 6: Análisis de Cobertura

**Concepto:** Días de demanda cubiertos por el stock de seguridad

**Fórmula:** `coverage_days = safety_stock / avg_demand`

**Interpretación:**
- **2–4 días**: Cobertura razonable para retail con lead time semanal
- **< 2 días**: Riesgo de ruptura ante picos de demanda
- **> 7 días**: Capital inmovilizado; revisar política o negociar lead time

**Caso de uso:** Ajustar safety stock si cobertura está fuera de rangos aceptables

**Decisión:** Balancear protección contra stockouts vs costo de oportunidad del capital

In [24]:
# Top 10 productos con mayor safety stock
top_safety = demand_stats.nlargest(10, 'safety_stock')

fig = px.bar(
    top_safety,
    x='sku',
    y='safety_stock',
    color='category',
    title="Top 10 Productos por Stock de Seguridad",
    labels={'safety_stock': 'Safety Stock (unidades)', 'sku': 'SKU'}
)
fig.update_xaxes(tickangle=-45)
fig.show()

# Distribución de días de cobertura
fig2 = px.histogram(
    demand_stats, x='coverage_days',
    title="Distribución de Días de Cobertura del Safety Stock",
    labels={'coverage_days': 'Días de Cobertura'},
    nbins=20
)
fig2.show()

print(f"📊 Cobertura promedio: {demand_stats['coverage_days'].mean():.1f} días")
print(f"📊 Cobertura mediana: {demand_stats['coverage_days'].median():.1f} días")

📊 Cobertura promedio: 3.4 días
📊 Cobertura mediana: 3.4 días


---

## 🔍 Paso 7: Comparar con Inventario Actual

**Concepto:** Identificar SKUs que requieren reposición urgente

**Métricas:**
- `needs_replenishment`: booleano (stock actual < reorder point)
- `stock_gap = max(0, reorder_point - current_stock)`

**Decisiones de negocio:**
- **Gap alto (>100 unidades)**: Orden urgente, riesgo de stockout inminente
- **Gap medio (20-100)**: Consolidar en próximo ciclo de reposición
- **Sin gap**: Mantener niveles actuales, monitorear evolución

**Caso de uso:** Priorizar reabastecimiento por criticidad y gap de inventario

In [25]:
# Unir con inventario actual (on_hand por SKU)
inventory_comparison = demand_stats.merge(
    df_inventory.groupby('sku')['on_hand'].sum().reset_index(),
    on='sku',
    how='left'
)
inventory_comparison.rename(columns={'on_hand': 'current_stock'}, inplace=True)
inventory_comparison['current_stock'].fillna(0, inplace=True)

# Identificar productos con stock insuficiente
inventory_comparison['needs_replenishment'] = (
    inventory_comparison['current_stock'] < inventory_comparison['reorder_point']
)

# Gap de inventario
inventory_comparison['stock_gap'] = (
    inventory_comparison['reorder_point'] - inventory_comparison['current_stock']
).clip(lower=0)

# Días de inventario actuales
inventory_comparison['days_on_hand'] = (
    inventory_comparison['current_stock'] / inventory_comparison['avg_demand']
).round(1)

# Clasificar urgencia de reabastecimiento
def classify_urgency(row):
    if not row['needs_replenishment']:
        return 'OK'
    elif row['stock_gap'] > row['safety_stock'] * 1.5:
        return 'Urgente'
    elif row['stock_gap'] > row['safety_stock'] * 0.5:
        return 'Alto'
    else:
        return 'Medio'

inventory_comparison['urgency'] = inventory_comparison.apply(classify_urgency, axis=1)

print("🔍 Comparación con Inventario Actual:")
display(inventory_comparison[[
    'sku', 'category', 'current_stock', 'days_on_hand', 
    'safety_stock', 'reorder_point', 'stock_gap', 'urgency'
]].head(10))

# Resumen ejecutivo
need_replen = inventory_comparison['needs_replenishment'].sum()
pct_need_replen = (need_replen / len(inventory_comparison)) * 100

print(f"\n📊 RESUMEN DE REABASTECIMIENTO:")
print(f"="*60)
print(f"   • SKUs totales analizados: {len(inventory_comparison)}")
print(f"   • SKUs que necesitan reorden: {need_replen} ({pct_need_replen:.1f}%)")
print(f"   • Gap total de inventario: {inventory_comparison['stock_gap'].sum():,.0f} unidades")
print(f"   • Valor promedio de gap: {inventory_comparison['stock_gap'].mean():.1f} unidades/SKU")

print(f"\n🚨 Priorización por Urgencia:")
urgency_counts = inventory_comparison['urgency'].value_counts()
for urgency, count in urgency_counts.items():
    pct = (count / len(inventory_comparison)) * 100
    emoji = '🔴' if urgency == 'Urgente' else '🟡' if urgency == 'Alto' else '🟢' if urgency == 'Medio' else '✅'
    print(f"   {emoji} {urgency}: {count} SKUs ({pct:.1f}%)")

# Top 10 SKUs con mayor gap
print(f"\n🔝 Top 10 SKUs con Mayor Gap de Inventario:")
top_gaps = inventory_comparison.nlargest(10, 'stock_gap')
for idx, row in top_gaps.iterrows():
    print(f"   {row['sku']:10} | {row['category']:15} | Gap: {row['stock_gap']:>6.0f} unidades | Urgencia: {row['urgency']}")

print(f"\n💼 Recomendaciones de Acción:")
urgent_count = len(inventory_comparison[inventory_comparison['urgency'] == 'Urgente'])
if urgent_count > 0:
    print(f"   1. 🔴 PRIORIDAD ALTA: Generar órdenes inmediatas para {urgent_count} SKUs urgentes")
    print(f"      → Riesgo inminente de stockout en próximos {LEAD_TIME_DAYS} días")

high_count = len(inventory_comparison[inventory_comparison['urgency'] == 'Alto'])
if high_count > 0:
    print(f"   2. 🟡 PRIORIDAD MEDIA: Programar reabastecimiento para {high_count} SKUs de prioridad alta")
    print(f"      → Consolidar en siguiente ciclo de órdenes (próximos 2-3 días)")

medium_count = len(inventory_comparison[inventory_comparison['urgency'] == 'Medio'])
if medium_count > 0:
    print(f"   3. 🟢 MONITOREO: Vigilar {medium_count} SKUs de prioridad media")
    print(f"      → Revisar en próxima revisión semanal de inventario")

ok_count = len(inventory_comparison[inventory_comparison['urgency'] == 'OK'])
print(f"   4. ✅ ESTABLE: {ok_count} SKUs tienen niveles de inventario adecuados")

🔍 Comparación con Inventario Actual:


,sku,category,current_stock,days_on_hand,safety_stock,reorder_point,stock_gap,urgency
0,SKU-00001,Household,945,73.1,41,132,0,OK
1,SKU-00002,Electronics,969,81.5,48,131,0,OK
2,SKU-00003,PersonalCare,724,66.4,35,111,0,OK
3,SKU-00004,Electronics,1380,94.5,46,148,0,OK
4,SKU-00005,Electronics,732,52.2,43,141,0,OK
5,SKU-00006,Snacks,769,64.5,34,118,0,OK
6,SKU-00007,PersonalCare,708,58.1,43,128,0,OK
7,SKU-00008,PersonalCare,987,92.7,37,112,0,OK
8,SKU-00009,PersonalCare,524,60.0,22,83,0,OK
9,SKU-00010,Electronics,1068,106.5,42,112,0,OK



📊 RESUMEN DE REABASTECIMIENTO:
   • SKUs totales analizados: 200
   • SKUs que necesitan reorden: 0 (0.0%)
   • Gap total de inventario: 0 unidades
   • Valor promedio de gap: 0.0 unidades/SKU

🚨 Priorización por Urgencia:
   ✅ OK: 200 SKUs (100.0%)

🔝 Top 10 SKUs con Mayor Gap de Inventario:
   SKU-00001  | Household       | Gap:      0 unidades | Urgencia: OK
   SKU-00002  | Electronics     | Gap:      0 unidades | Urgencia: OK
   SKU-00003  | PersonalCare    | Gap:      0 unidades | Urgencia: OK
   SKU-00004  | Electronics     | Gap:      0 unidades | Urgencia: OK
   SKU-00005  | Electronics     | Gap:      0 unidades | Urgencia: OK
   SKU-00006  | Snacks          | Gap:      0 unidades | Urgencia: OK
   SKU-00007  | PersonalCare    | Gap:      0 unidades | Urgencia: OK
   SKU-00008  | PersonalCare    | Gap:      0 unidades | Urgencia: OK
   SKU-00009  | PersonalCare    | Gap:      0 unidades | Urgencia: OK
   SKU-00010  | Electronics     | Gap:      0 unidades | Urgencia: OK

💼 Re

---

## 📈 Paso 8: Sensibilidad del Nivel de Servicio

**Concepto:** Análisis de impacto al variar el nivel de servicio objetivo

**Relación:** A mayor `service_level` → mayor `z_score` → mayor `safety_stock`

**Trade-off:**
- **Aumentar SL**: Menos stockouts, mayor satisfacción del cliente, pero más capital invertido
- **Reducir SL**: Menos inventario, mejor flujo de caja, pero más riesgo de rupturas

**Recomendación por clasificación ABC:**
- **Clase A (críticos, alto valor)**: 98%–99%
- **Clase B (moderados)**: 95%
- **Clase C (bajo valor/rotación)**: 90%–92%

**Caso de uso:** Evaluar sensibilidad antes de comprometer capital en inventario

In [26]:
# Analizar diferentes niveles de servicio
service_levels = [0.90, 0.95, 0.98, 0.99]
sample_sku_data = demand_stats.iloc[0]

sensitivity_results = []
for sl in service_levels:
    z = stats.norm.ppf(sl)
    ss = z * sample_sku_data['std_demand'] * np.sqrt(LEAD_TIME_DAYS)
    sensitivity_results.append({
        'service_level': f"{sl*100:.0f}%",
        'z_score': z,
        'safety_stock': int(ss)
    })

df_sensitivity = pd.DataFrame(sensitivity_results)

fig = px.bar(
    df_sensitivity,
    x='service_level',
    y='safety_stock',
    title=f"Sensibilidad del Safety Stock al Nivel de Servicio<br>SKU: {sample_sku_data['sku']}",
    labels={'service_level': 'Nivel de Servicio', 'safety_stock': 'Safety Stock (unidades)'},
    text='safety_stock'
)
fig.update_traces(textposition='outside')
fig.show()

print("📊 Análisis de Sensibilidad:")
display(df_sensitivity)

📊 Análisis de Sensibilidad:


,service_level,z_score,safety_stock
0,90%,1.281552,31
1,95%,1.644854,40
2,98%,2.053749,50
3,99%,2.326348,57


---

## 💾 Paso 9: Exportar Resultados

**Formato:** CSV para compatibilidad con sistemas downstream

**Outputs:**
- Políticas de inventario por SKU (safety stock, ROP, coverage)
- Resumen ejecutivo con KPIs agregados

In [27]:
# Guardar políticas de inventario
output_file = OUTPUT_DIR / "inventory_policies.csv"
inventory_comparison.to_csv(output_file, index=False)

print(f"💾 Políticas guardadas: {output_file}")
print(f"📏 Dimensiones: {inventory_comparison.shape}")

# Resumen ejecutivo
summary = {
    'total_skus': len(inventory_comparison),
    'avg_safety_stock': inventory_comparison['safety_stock'].mean(),
    'total_safety_stock': inventory_comparison['safety_stock'].sum(),
    'skus_need_replenishment': need_replen,
    'total_stock_gap': inventory_comparison['stock_gap'].sum(),
    'service_level': SERVICE_LEVEL,
    'lead_time_days': LEAD_TIME_DAYS
}

print("\n📋 RESUMEN EJECUTIVO")
print("="*50)
for key, value in summary.items():
    print(f"  {key}: {value}")

💾 Políticas guardadas: f:\GitHub\supply-chain-data-notebooks\data\processed\or07\inventory_policies.csv
📏 Dimensiones: (200, 17)

📋 RESUMEN EJECUTIVO
  total_skus: 200
  avg_safety_stock: 40.69
  total_safety_stock: 8138
  skus_need_replenishment: 0
  total_stock_gap: 0
  service_level: 0.95
  lead_time_days: 7


---

## 🛠️ Paso 10: Funciones Reutilizables

**Concepto:** Encapsular lógica de cálculo para reutilización en otros notebooks

**Beneficio:** Modularidad, consistencia en cálculos, facilita mantenimiento

In [28]:
def calculate_safety_stock(
    avg_demand: float,
    std_demand: float,
    service_level: float,
    lead_time_days: int
) -> dict:
    """
    Calcula stock de seguridad y punto de reorden.
    
    Args:
        avg_demand: Demanda promedio diaria
        std_demand: Desviación estándar de demanda diaria
        service_level: Nivel de servicio deseado (0-1)
        lead_time_days: Lead time de reabastecimiento (días)
    
    Returns:
        Dict con safety_stock, reorder_point, z_score, coverage_days
    """
    z_score = stats.norm.ppf(service_level)
    safety_stock = z_score * std_demand * np.sqrt(lead_time_days)
    reorder_point = (avg_demand * lead_time_days) + safety_stock
    
    return {
        'safety_stock': int(safety_stock),
        'reorder_point': int(reorder_point),
        'z_score': round(z_score, 2),
        'coverage_days': round(safety_stock / avg_demand, 1) if avg_demand > 0 else 0
    }

# Ejemplo de uso:
# result = calculate_safety_stock(avg_demand=50, std_demand=15, service_level=0.95, lead_time_days=7)
# print(result)

print("✅ Funciones reutilizables definidas")

✅ Funciones reutilizables definidas


---

## 📊 Paso 11: Políticas Diferenciadas por Categoría

**Concepto:** Ajustar parámetros (SL, LT) por categoría de producto

**Configuración propuesta:**
- **Electronics**: SL 98%, LT 10 días (productos críticos, lead time largo)
- **PersonalCare**: SL 95%, LT 7 días (estándar)
- **Household**: SL 92%, LT 5 días (menor criticidad)
- **Snacks**: SL 90%, LT 4 días (perecederos, rotación rápida)

**Objetivo:** Optimizar capital de trabajo asignando recursos según criticidad y valor

In [29]:
# Parámetros por categoría
category_params = {
    'Electronics': {'SERVICE_LEVEL': 0.98, 'LEAD_TIME_DAYS': 10},
    'PersonalCare': {'SERVICE_LEVEL': 0.95, 'LEAD_TIME_DAYS': 7},
    'Household': {'SERVICE_LEVEL': 0.92, 'LEAD_TIME_DAYS': 5},
    'Snacks': {'SERVICE_LEVEL': 0.90, 'LEAD_TIME_DAYS': 4},
    'Unknown': {'SERVICE_LEVEL': SERVICE_LEVEL, 'LEAD_TIME_DAYS': LEAD_TIME_DAYS},
}

# Asegurar que df_products tenga columna 'category'
if 'category' not in df_products.columns:
    df_products = df_products.copy()
    df_products['category'] = 'Unknown'

# Función para calcular políticas por fila
def compute_policy_row(row):
    cat = row.get('category', 'Unknown')
    params = category_params.get(cat, category_params['Unknown'])
    sl = params['SERVICE_LEVEL']
    lt = params['LEAD_TIME_DAYS']
    z = stats.norm.ppf(sl)
    sigma = row['std_demand'] if pd.notnull(row['std_demand']) else 0.0
    avg = row['avg_demand'] if pd.notnull(row['avg_demand']) else 0.0
    safety_stock = z * sigma * np.sqrt(lt)
    reorder_point = avg * lt + safety_stock
    coverage_days = (safety_stock / avg) if avg > 0 else 0.0
    return pd.Series({
        'service_level_cat': sl,
        'lead_time_days_cat': lt,
        'z_score_cat': round(z, 2),
        'safety_stock_cat': int(safety_stock),
        'reorder_point_cat': int(reorder_point),
        'coverage_days_cat': round(coverage_days, 1),
    })

# Preparar dataframe con categoría
demand_stats_cat = inventory_comparison.copy()
if 'category' not in demand_stats_cat.columns:
    demand_stats_cat = demand_stats_cat.merge(
        df_products[['sku', 'category']], 
        on='sku', 
        how='left'
    )
    demand_stats_cat['category'].fillna('Unknown', inplace=True)

# Aplicar cálculo por fila
computed = demand_stats_cat.apply(compute_policy_row, axis=1)
for col in computed.columns:
    demand_stats_cat[col] = computed[col]

print('📏 Políticas por categoría calculadas:')
display(demand_stats_cat[[
    'sku', 'category', 'avg_demand', 'std_demand', 
    'service_level_cat', 'lead_time_days_cat', 
    'safety_stock_cat', 'reorder_point_cat', 'coverage_days_cat'
]].head())

# Exportar CSV
output_file_cat = OUTPUT_DIR / 'inventory_policies_by_category.csv'
demand_stats_cat.to_csv(output_file_cat, index=False)
print(f'✅ Exportado: {output_file_cat}')

📏 Políticas por categoría calculadas:


,sku,category,avg_demand,std_demand,service_level_cat,lead_time_days_cat,safety_stock_cat,reorder_point_cat,coverage_days_cat
0,SKU-00001,Household,12.933333,9.373232,0.92,5.0,29.0,94.0,2.3
1,SKU-00002,Electronics,11.888889,11.000721,0.98,10.0,71.0,190.0,6.0
2,SKU-00003,PersonalCare,10.909091,8.063667,0.95,7.0,35.0,111.0,3.2
3,SKU-00004,Electronics,14.604651,10.659535,0.98,10.0,69.0,215.0,4.7
4,SKU-00005,Electronics,14.024390,9.903251,0.98,10.0,64.0,204.0,4.6


✅ Exportado: f:\GitHub\supply-chain-data-notebooks\data\processed\or07\inventory_policies_by_category.csv


In [30]:
# Guardar artefactos de visualización (HTML)
try:
    # Figura: Top 10 Safety Stock por categoría
    if 'demand_stats_cat' in globals():
        top_safety_cat = demand_stats_cat.nlargest(10, 'safety_stock_cat')
        fig_top = px.bar(
            top_safety_cat, 
            x='sku', 
            y='safety_stock_cat', 
            color='category',
            title='Top 10 SKUs por Safety Stock (Políticas por Categoría)',
            labels={'safety_stock_cat': 'Safety Stock', 'sku': 'SKU'}
        )
        fig_top.update_xaxes(tickangle=-45)
        html_top = OUTPUT_DIR / 'top10_safety_stock_by_category.html'
        fig_top.write_html(html_top)
        print(f'✅ HTML exportado: {html_top}')
    
    # Figura: Distribución de Cobertura por Categoría
    if 'demand_stats_cat' in globals() and 'coverage_days_cat' in demand_stats_cat.columns:
        fig_cov = px.histogram(
            demand_stats_cat, 
            x='coverage_days_cat', 
            color='category',
            nbins=20, 
            title='Distribución de Cobertura en Días (por Categoría)',
            labels={'coverage_days_cat': 'Días de Cobertura'}
        )
        html_cov = OUTPUT_DIR / 'coverage_days_by_category.html'
        fig_cov.write_html(html_cov)
        print(f'✅ HTML exportado: {html_cov}')
except Exception as e:
    print(f'⚠️ Error al generar visualizaciones: {e}')

print(f"\n📂 Todos los artefactos guardados en: {OUTPUT_DIR}")

✅ HTML exportado: f:\GitHub\supply-chain-data-notebooks\data\processed\or07\top10_safety_stock_by_category.html
✅ HTML exportado: f:\GitHub\supply-chain-data-notebooks\data\processed\or07\coverage_days_by_category.html

📂 Todos los artefactos guardados en: f:\GitHub\supply-chain-data-notebooks\data\processed\or07


---

## 📋 Paso 12: Resumen Ejecutivo

**Concepto:** Consolidación de KPIs clave y recomendaciones de negocio

In [31]:
# Resumen ejecutivo con KPIs
print("\n" + "="*60)
print("📊 RESUMEN EJECUTIVO - SAFETY STOCK ANALYSIS")
print("="*60)

# KPIs generales
print("\n🔢 KPIs Generales:")
print(f"  • Total SKUs analizados: {len(demand_stats_cat) if 'demand_stats_cat' in globals() else len(demand_stats)}")
print(f"  • Safety Stock total (baseline): {demand_stats['safety_stock'].sum():,.0f} unidades")
if 'demand_stats_cat' in globals():
    print(f"  • Safety Stock total (por categoría): {demand_stats_cat['safety_stock_cat'].sum():,.0f} unidades")
print(f"  • SKUs que requieren reposición: {need_replen} ({need_replen/len(inventory_comparison)*100:.1f}%)")
print(f"  • Gap total de inventario: {inventory_comparison['stock_gap'].sum():,.0f} unidades")

# KPIs por categoría
if 'demand_stats_cat' in globals() and 'category' in demand_stats_cat.columns:
    print("\n📦 KPIs por Categoría:")
    cat_summary = demand_stats_cat.groupby('category').agg({
        'sku': 'count',
        'safety_stock_cat': 'sum',
        'coverage_days_cat': 'mean',
        'cv': 'mean'
    }).round(1)
    cat_summary.columns = ['SKUs', 'Safety Stock Total', 'Cobertura Promedio (días)', 'CV Promedio']
    display(cat_summary)

# Recomendaciones
print("\n💡 Recomendaciones de Negocio:")
high_cv_skus = len(demand_stats[demand_stats['cv'] > 0.8])
print(f"  1. 🔴 Revisar {high_cv_skus} SKUs con CV > 0.8 (alta variabilidad)")
print(f"  2. ⚡ Priorizar reabastecimiento de {need_replen} SKUs con gap de inventario")
if 'demand_stats_cat' in globals():
    electronics = demand_stats_cat[demand_stats_cat['category'] == 'Electronics']
    if len(electronics) > 0 and electronics['coverage_days_cat'].mean() < 3:
        print(f"  3. 📈 Considerar aumentar SL a 98-99% para Electronics (cobertura < 3 días)")
print(f"  4. 💰 Evaluar reducción de SL para productos de baja rotación (Clase C)")

print("\n📂 Artefactos generados:")
print(f"  • CSV: {OUTPUT_DIR}/inventory_policies.csv")
if 'demand_stats_cat' in globals():
    print(f"  • CSV: {OUTPUT_DIR}/inventory_policies_by_category.csv")
print(f"  • HTML: {OUTPUT_DIR}/*.html")
print("\n✅ Análisis completado")


📊 RESUMEN EJECUTIVO - SAFETY STOCK ANALYSIS

🔢 KPIs Generales:
  • Total SKUs analizados: 200
  • Safety Stock total (baseline): 8,138 unidades
  • Safety Stock total (por categoría): 7,601 unidades
  • SKUs que requieren reposición: 0 (0.0%)
  • Gap total de inventario: 0 unidades

📦 KPIs por Categoría:


,SKUs,Safety Stock Total,Cobertura Promedio (días),CV Promedio
category,,,,
Beverages,43,1687.0,3.3,0.8
Electronics,36,2125.0,5.0,0.8
Household,49,1400.0,2.4,0.8
PersonalCare,37,1553.0,3.5,0.8
Snacks,35,836.0,2.0,0.8



💡 Recomendaciones de Negocio:
  1. 🔴 Revisar 79 SKUs con CV > 0.8 (alta variabilidad)
  2. ⚡ Priorizar reabastecimiento de 0 SKUs con gap de inventario
  4. 💰 Evaluar reducción de SL para productos de baja rotación (Clase C)

📂 Artefactos generados:
  • CSV: f:\GitHub\supply-chain-data-notebooks\data\processed\or07/inventory_policies.csv
  • CSV: f:\GitHub\supply-chain-data-notebooks\data\processed\or07/inventory_policies_by_category.csv
  • HTML: f:\GitHub\supply-chain-data-notebooks\data\processed\or07/*.html

✅ Análisis completado


---

## ✅ Validaciones

In [32]:
# Validaciones de integridad y lógica de negocio
assert len(demand_stats) > 0, "demand_stats no debe estar vacío"
assert demand_stats['safety_stock'].min() >= 0, "Safety stock debe ser no negativo"
assert demand_stats['reorder_point'].min() >= 0, "Reorder point debe ser no negativo"
assert demand_stats['cv'].notna().all(), "CV no debe tener valores nulos"
assert (demand_stats['avg_demand'] >= 0).all(), "Demanda promedio debe ser no negativa"

if 'demand_stats_cat' in globals():
    assert len(demand_stats_cat) > 0, "demand_stats_cat no debe estar vacío"
    assert demand_stats_cat['safety_stock_cat'].min() >= 0, "Safety stock por categoría debe ser no negativo"
    assert demand_stats_cat['service_level_cat'].between(0, 1).all(), "Service level debe estar entre 0 y 1"

print("✅ Validaciones pasadas")
print(f"✅ Notebook OR-07 completado: Análisis de Safety Stock con {len(demand_stats)} SKUs")

✅ Validaciones pasadas
✅ Notebook OR-07 completado: Análisis de Safety Stock con 200 SKUs


---

## 📚 Resumen Técnico y Referencias

### 🎯 Resultados Clave

Este notebook implementa el cálculo estadístico de **Stock de Seguridad** para proteger contra variabilidad de demanda durante el lead time de reabastecimiento.

**Componentes/Métricas calculadas:**
1. **Coeficiente de Variación (CV)**: `CV = σ / μ` - Medida de variabilidad relativa de demanda
2. **Safety Stock**: `SS = Z × σ × √LT` - Buffer de inventario basado en nivel de servicio
3. **Reorder Point (ROP)**: `ROP = μ × LT + SS` - Punto que dispara reabastecimiento
4. **Días de Cobertura**: `Coverage = SS / μ` - Días de demanda cubiertos por el safety stock

**Hallazgos típicos:**
- Productos con CV > 0.7 requieren safety stock significativamente mayor
- Aumentar nivel de servicio de 95% a 99% incrementa SS en ~40%
- Políticas diferenciadas por categoría optimizan capital de trabajo

**Segmentación por Nivel de Servicio:**
- **Clase A (críticos)**: SL 98-99% → Mayor protección, menor riesgo de stockout
- **Clase B (moderados)**: SL 95% → Balance estándar costo-servicio
- **Clase C (bajo valor)**: SL 90-92% → Menor inversión en inventario

### 🔬 Metodología

**Modelo/Fórmula principal:**

$$
\text{Safety Stock} = Z_{\alpha} \times \sigma_{\text{demanda}} \times \sqrt{\text{Lead Time}}
$$

$$
\text{Reorder Point} = \mu_{\text{demanda}} \times \text{Lead Time} + \text{Safety Stock}
$$

$$
\text{Coeficiente de Variación} = \frac{\sigma_{\text{demanda}}}{\mu_{\text{demanda}}}
$$

**Técnica aplicada:**
- Distribución normal estándar para modelar demanda
- Z-score (cuantil normal) para nivel de servicio: `Z = Φ⁻¹(service_level)`
- Supuesto: Demanda es independiente e idénticamente distribuida (i.i.d.)
- Parámetros: SL típico 95% (Z=1.65), LT 4-10 días según categoría

### 📖 Aplicaciones Prácticas

1. **Auditoría de Políticas Existentes:**
   - Comparar safety stock actual vs calculado estadísticamente
   - Identificar productos sobre/sub-inventariados

2. **Planificación de Compras:**
   - Disparar órdenes de compra cuando inventario < ROP
   - Priorizar reabastecimiento por gap de inventario

3. **Optimización de Capital de Trabajo:**
   - Ajustar niveles de servicio por clasificación ABC
   - Reducir inventario excedente sin comprometer servicio

### 🔗 Referencias

1. **Silver, E. A., Pyke, D. F., & Peterson, R. (1998)**. *Inventory Management and Production Planning and Scheduling*. Wiley.
   - Fundamentos de políticas de inventario (Q,R) y safety stock clásico

2. **Chopra, S., & Meindl, P. (2016)**. *Supply Chain Management: Strategy, Planning, and Operation*. Pearson.
   - Trade-offs entre costo de inventario y nivel de servicio

3. **Nahmias, S., & Cheng, Y. (2009)**. *Production and Operations Analysis*. McGraw-Hill.
   - Modelos estocásticos de inventario bajo incertidumbre de demanda

### 💡 Extensiones Futuras

- Incluir variabilidad de lead time (σ_LT) en la fórmula de safety stock
- Implementar políticas (s,S) con revisión periódica
- Optimización multi-echelon (centro de distribución + tiendas)
- Análisis de estacionalidad con modelos SARIMA
- Machine Learning para pronóstico de demanda y cálculo dinámico de SS

---

**Autor**: lraigosov (@LuisRai)  
**Fecha**: 2024 a la actualidad  
**Versión**: 1.0  
**Tags**: `#safety-stock` `#inventory-optimization` `#service-level` `#operations-research`

---

<div style="width: 100%; clear: both; margin: 0 0 20px 0; border-top: 1px solid #eaecef; padding-top: 24px;"><div style="display: flex; justify-content: space-between; align-items: center; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif;"><div style="flex: 1; text-align: left;"><a href="OR-06-dock_queue_simulation.ipynb" style="text-decoration: none; color: #0366d6; font-size: 14px; font-weight: 600; transition: color 0.2s;">← Anterior: OR-06-dock_queue_simulation.ipynb</a></div><div style="flex: 1; text-align: center; font-size: 14px;"><a href="../../README.md" style="color: #0366d6; text-decoration: none; font-weight: 600; margin: 0 10px;">📑 Índice</a><span style="color: #6a737d;">|</span><a href="../../config/notebooks_index.yml" style="color: #0366d6; text-decoration: none; font-weight: 600; margin: 0 10px;">📋 Catálogo</a></div><div style="flex: 1; text-align: right;"><a href="OR-08-production_scheduling.ipynb" style="text-decoration: none; color: #0366d6; font-size: 14px; font-weight: 600; transition: color 0.2s;">Siguiente: OR-08-production_scheduling.ipynb →</a></div></div></div>